<a href="https://colab.research.google.com/github/Shubhamspc90/GenAI-Using-LangChain/blob/lang/Agent/building%20agent/5__Weather_Assistant_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-ollama requests

In [3]:
# fetching API key
from google.colab import userdata
ollama_api_key = userdata.get('OLLAMA_API_KEY')
print("API key Found : ",bool(ollama_api_key))


API key Found :  True


In [8]:
# creating model
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model = "gpt-oss:20b-cloud",
    base_url = "https://ollama.com",
    client_kwargs = {
        "headers":{
            "Authorization": f"Bearer {ollama_api_key}"
        }
    },
    temperature = 0.1
)

In [9]:
# testing Model Working
print(llm.invoke("hi"))

content='Hello! How can I help you today?' additional_kwargs={} response_metadata={'model': 'gpt-oss:20b-cloud', 'created_at': '2026-09-24T09:40:41.123235753Z', 'done': True, 'done_reason': 'stop', 'total_duration': 786556520, 'load_duration': None, 'prompt_eval_count': 68, 'prompt_eval_duration': None, 'eval_count': 36, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:20b-cloud', 'model_provider': 'ollama'} id='lc_run--01a0d2c9-de92-7770-84df-8358833aed5a-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 68, 'output_tokens': 36, 'total_tokens': 104}


In [12]:
# creating custom tool : (Weather Tool)
import requests

from langchain_core.tools import tool

@tool
def get_weather(city: str) -> dict:
    """Get current weather information for a city."""

    # Step 1: City ka latitude aur longitude find karna
    geo_url = "https://geocoding-api.open-meteo.com/v1/search"

    geo_response = requests.get(
        geo_url,
        params={
            "name": city,
            "count": 1,
            "language": "en",
            "format": "json"
        }
    )

    geo_data = geo_response.json()

    # Agar city nahi mili
    if "results" not in geo_data:
        return {
            "error": f"City '{city}' not found"
        }

    # City ki location information
    latitude = geo_data["results"][0]["latitude"]
    longitude = geo_data["results"][0]["longitude"]

    # Step 2: Weather API se current weather lena
    weather_url = "https://api.open-meteo.com/v1/forecast"

    weather_response = requests.get(
        weather_url,
        params={
            "latitude": latitude,
            "longitude": longitude,
            "current": "temperature_2m,relative_humidity_2m,wind_speed_10m"
        }
    )

    weather_data = weather_response.json()

    # Current weather data
    current = weather_data["current"]

    # Tool ka final output
    return {
        "city": city,
        "temperature": current["temperature_2m"],
        "humidity": current["relative_humidity_2m"],
        "wind_speed": current["wind_speed_10m"]
    }

In [13]:
# creating Agent
from langchain.agents import create_agent
agent = create_agent(
    model =llm,
    tools = [get_weather]
)

In [14]:
# user Query
user_question ="What is the current weather in Noida"

In [15]:
# tool calling
result = agent.invoke({
     "messages" : [
         {
             "role" : "user",
             "content": user_question
         }
     ]
})


In [16]:
#result
print(result["messages"][-1].content)

**Current Weather in Noida**

- **Temperature:** 34.4 °C  
- **Humidity:** 40 %  
- **Wind Speed:** 5.4 km/h  

Let me know if you’d like more details (e.g., precipitation, UV index, or forecast)!
